In [ ]:
!pip install torch torchvision tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cuda


Обучаем архитектуру, которая далее будет считаться Учителем.
Обучаем архитектуру, которая далее будет считаться Студентом.

Загружаем датасет

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
train_loader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:05<00:00, 31.1MB/s]


Имплементируем модели

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(128, 10)

    def forward(self, x, return_feature=False):
        f = self.features(x)
        f = f.view(f.size(0), -1)
        out = self.fc(f)
        return (out, f) if return_feature else out

# Teacher — ResNet18
def build_teacher():
    model = torchvision.models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(512, 10)
    return model

teacher = build_teacher().to(device)
student = SmallCNN().to(device)

Обучаем модели

In [ ]:
def train_model(model, train_loader, test_loader, epochs=10, lr=0.1, name="model"):
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=7, gamma=0.1)

    for epoch in range(epochs):
        model.train()
        total, correct, total_loss = 0, 0, 0
        for x, y in tqdm(train_loader, desc=f"{name} epoch {epoch+1}/{epochs}"):
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            out = model(x)
            loss = F.cross_entropy(out, y)
            loss.backward()
            opt.step()
            total_loss += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
        scheduler.step()
        acc = correct / total
        print(f"Train acc={acc:.3f}, loss={total_loss/total:.3f}")

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    print(f"{name} test acc = {correct/total:.3f}")
    torch.save(model.state_dict(), f"{name}.pth")
    print(f"Saved weights to {name}.pth")

# --- Обучаем Teacher и Student ---
train_model(teacher, train_loader, test_loader, epochs=10, lr=0.1, name="teacher")
train_model(student, train_loader, test_loader, epochs=10, lr=0.1, name="student_base")

teacher epoch 1/10: 100%|██████████| 391/391 [00:41<00:00,  9.51it/s]


Train acc=0.289, loss=1.976


teacher epoch 2/10: 100%|██████████| 391/391 [00:42<00:00,  9.21it/s]


Train acc=0.473, loss=1.437


teacher epoch 3/10: 100%|██████████| 391/391 [00:42<00:00,  9.29it/s]


...

student_base epoch 9/10: 100%|██████████| 391/391 [00:18<00:00, 20.81it/s]


Train acc=0.747, loss=0.736


student_base epoch 10/10: 100%|██████████| 391/391 [00:18<00:00, 21.66it/s]

Train acc=0.751, loss=0.720


student_base test acc = 0.736
Saved weights to student_base.pth


Метрика accuracy на тесте у Учителя 0.876, у Ученика равна  0.736

**Эксперимент 1 - Дистилляция логитов**

Измените тренировочный цикл так, чтобы параллельно в две архитектуры подавались семплы, при этом в Учителе должно быть отколючено обновление градиентов (torch.no_grad()).
Изменить функцию ошибки.

Загружаем предобученные модели Учителя и ученика, создаём функцию дистилляции

In [ ]:
teacher = build_teacher().to(device)
student = SmallCNN().to(device)

teacher.load_state_dict(torch.load("teacher.pth", map_location=device))
student.load_state_dict(torch.load("student_base.pth", map_location=device))
print("✅ Загружены teacher.pth и student_base.pth")

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

def distill_train_epoch(teacher, student, loader, optimizer, T=4.0, alpha=0.5):
    student.train()
    total_loss, total, correct = 0, 0, 0
    for x, y in tqdm(loader, desc="KD (logits)"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        with torch.no_grad():
            t_logits = teacher(x)

        s_logits = student(x)

        ce_loss = F.cross_entropy(s_logits, y)
        p_t = F.softmax(t_logits / T, dim=1)
        p_s = F.log_softmax(s_logits / T, dim=1)
        kl = F.kl_div(p_s, p_t, reduction='batchmean') * (T * T)

        loss = alpha * ce_loss + (1 - alpha) * kl
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)
        correct += (s_logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total

✅ Загружены teacher.pth и student_base.pth


Обучаем, считаем метрики

In [ ]:
@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = F.cross_entropy(out, y)
        loss_sum += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, correct / total

opt = torch.optim.SGD(student.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5)

best_acc = 0
for epoch in range(10):
    train_loss, train_acc = distill_train_epoch(teacher, student, train_loader, opt, T=4.0, alpha=0.5)
    val_loss, val_acc = eval_model(student, test_loader)
    scheduler.step()
    print(f"Epoch {epoch+1}: train_loss={train_loss:.3f}, acc={train_acc:.3f}, val_acc={val_acc:.3f}")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(student.state_dict(), "student_logits_distilled.pth")

print("✅ Best val acc:", best_acc)

KD (logits): 100%|██████████| 391/391 [00:20<00:00, 19.42it/s]


Epoch 1: train_loss=1.455, acc=0.720, val_acc=0.730


KD (logits): 100%|██████████| 391/391 [00:19<00:00, 19.90it/s]


Epoch 2: train_loss=1.294, acc=0.738, val_acc=0.746


KD (logits): 100%|██████████| 391/391 [00:20<00:00, 18.77it/s]


...

Epoch 8: train_loss=0.840, acc=0.805, val_acc=0.794


KD (logits): 100%|██████████| 391/391 [00:20<00:00, 19.50it/s]


Epoch 9: train_loss=0.841, acc=0.805, val_acc=0.792


KD (logits): 100%|██████████| 391/391 [00:19<00:00, 20.03it/s]


Epoch 10: train_loss=0.821, acc=0.807, val_acc=0.793
✅ Best val acc: 0.7939


**Эксперимент 2 - Учим Студента совпадать по скрытому состоянию с Учителем (без модификации и обучения архитектур)**

Привести какие-либо блоки скрытого пространства Учителя и Студента к одной размерности.
Добавить в целевую функцию ошибки cosine loss между фичами Студента и Учителя.

Загружаем модели, создаем функцию ошибки cosine loss



In [ ]:

teacher.load_state_dict(torch.load("teacher.pth", map_location=device))
student.load_state_dict(torch.load("student_base.pth", map_location=device))
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False
print("Loaded teacher & student weights")

proj_teacher = nn.Linear(512, 128).to(device)

def cosine_loss(a, b):
    a_n = F.normalize(a, dim=1)
    b_n = F.normalize(b, dim=1)
    return (1 - (a_n * b_n).sum(dim=1)).mean()

Loaded teacher & student weights


 Извлекаем фичи учителя (до FC)

In [ ]:
class TeacherFeatureExtractor(nn.Module):
    def __init__(self, teacher):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            teacher.conv1,
            teacher.bn1,
            teacher.relu,
            teacher.layer1,
            teacher.layer2,
            teacher.layer3,
            teacher.layer4,
            teacher.avgpool
        )
    def forward(self, x):
        f = self.feature_extractor(x)
        f = f.view(f.size(0), -1)
        return f

teacher_feat_extractor = TeacherFeatureExtractor(teacher).to(device)
teacher_feat_extractor.eval()

TeacherFeatureExtractor(
  (feature_extractor): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2

Обучаем и фиксируем метрики

In [ ]:
def train_cosine_kd(teacher_feat_extractor, student, loader, optimizer, gamma=1.0):
    student.train()
    total_loss, total, correct = 0, 0, 0
    for x, y in tqdm(loader, desc="Cosine KD"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        # Получаем фичи учителя
        with torch.no_grad():
            t_feat = teacher_feat_extractor(x)
            t_feat = proj_teacher(t_feat)  # 🔁 Приводим к размерности 128

        # Получаем фичи студента
        s_out, s_feat = student(x, return_feature=True)

        ce = F.cross_entropy(s_out, y)
        feat_loss = cosine_loss(s_feat, t_feat)
        loss = ce + gamma * feat_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)
        correct += (s_out.argmax(1) == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total

@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = F.cross_entropy(out, y)
        loss_sum += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, correct / total

opt = torch.optim.SGD(list(student.parameters()) + list(proj_teacher.parameters()),
                      lr=0.05, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5)

best_acc = 0
for epoch in range(10):
    train_loss, train_acc = train_cosine_kd(teacher_feat_extractor, student, train_loader, opt, gamma=1.0)
    val_loss, val_acc = eval_model(student, test_loader)
    scheduler.step()
    print(f"Epoch {epoch+1}: train_loss={train_loss:.3f}, acc={train_acc:.3f}, val_acc={val_acc:.3f}")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(student.state_dict(), "student_cosine_distilled.pth")

print("✅ Best val acc:", best_acc)

Cosine KD: 100%|██████████| 391/391 [00:20<00:00, 19.03it/s]


Epoch 1: train_loss=1.523, acc=0.702, val_acc=0.713


Cosine KD: 100%|██████████| 391/391 [00:22<00:00, 17.13it/s]


Epoch 2: train_loss=1.448, acc=0.710, val_acc=0.715


Cosine KD: 100%|██████████| 391/391 [00:20<00:00, 19.49it/s]


...

Epoch 8: train_loss=1.241, acc=0.766, val_acc=0.765


Cosine KD: 100%|██████████| 391/391 [00:19<00:00, 19.66it/s]


Epoch 9: train_loss=1.236, acc=0.767, val_acc=0.743


Cosine KD: 100%|██████████| 391/391 [00:20<00:00, 18.93it/s]


Epoch 10: train_loss=1.221, acc=0.772, val_acc=0.756
✅ Best val acc: 0.7655


**Эксперимент 3 - Добавляем обучаемый регрессор**

Добавить в архитектуру новый блок, который может обучаться (conv2d).
Добавить в целевую функцию ошибки MSE loss между фичами Студента и Учителя.

Загружаем обученные модели из пререквизита

In [ ]:
teacher = build_teacher().to(device)
student = SmallCNN().to(device)

teacher.load_state_dict(torch.load("teacher.pth", map_location=device))
student.load_state_dict(torch.load("student_base.pth", map_location=device))
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False
print("✅ Teacher & Student загружены")

✅ Teacher & Student загружены


Извлечение фич учителя

In [ ]:
class TeacherFeatureExtractor(nn.Module):
    def __init__(self, teacher):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            teacher.conv1,
            teacher.bn1,
            teacher.relu,
            teacher.layer1,
            teacher.layer2,
            teacher.layer3,
            teacher.layer4,
            teacher.avgpool
        )
    def forward(self, x):
        f = self.feature_extractor(x)
        return f  # shape: (B, 512, 1, 1)

teacher_feat_extractor = TeacherFeatureExtractor(teacher).to(device)
teacher_feat_extractor.eval()


TeacherFeatureExtractor(
  (feature_extractor): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2

Добавляем обучаемый регрессор (Conv1x1) и создаем MSE Distillation Loss

In [ ]:
class StudentWithRegressor(nn.Module):
    def __init__(self, base_student, teacher_channels=512):
        super().__init__()
        self.student = base_student
        # регрессор для перевода фичей студента (128 каналов) в пространство учителя (512)
        self.regressor = nn.Conv2d(128, teacher_channels, kernel_size=1)

    def forward(self, x):
        # получаем фичи студента
        f = self.student.features(x)
        f_reg = self.regressor(f)
        out = self.student.fc(f.view(f.size(0), -1))
        return out, f_reg

student_reg = StudentWithRegressor(student).to(device)

def distill_mse_train_epoch(teacher_feat_extractor, student_reg, loader, optimizer, gamma=1.0):
    student_reg.train()
    total_loss, total, correct = 0, 0, 0
    for x, y in tqdm(loader, desc="MSE KD"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        # фичи учителя
        with torch.no_grad():
            t_feat = teacher_feat_extractor(x)

        # фичи студента (через регрессор)
        s_out, s_feat_reg = student_reg(x)

        ce_loss = F.cross_entropy(s_out, y)
        feat_loss = F.mse_loss(s_feat_reg, t_feat)
        loss = ce_loss + gamma * feat_loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)
        correct += (s_out.argmax(1) == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total


Обучаем и фиксируем метрики

In [ ]:
@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out, _ = model(x)
        loss = F.cross_entropy(out, y)
        loss_sum += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, correct / total


# --- Обучение ---
opt = torch.optim.SGD(student_reg.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5)

best_acc = 0
for epoch in range(10):
    train_loss, train_acc = distill_mse_train_epoch(teacher_feat_extractor, student_reg, train_loader, opt, gamma=1.0)
    val_loss, val_acc = eval_model(student_reg, test_loader)
    scheduler.step()
    print(f"Epoch {epoch+1}: train_loss={train_loss:.3f}, acc={train_acc:.3f}, val_acc={val_acc:.3f}")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(student_reg.state_dict(), "student_mse_regressor.pth")

print("✅ Best val acc:", best_acc)

MSE KD: 100%|██████████| 391/391 [00:21<00:00, 18.12it/s]


Epoch 1: train_loss=0.849, acc=0.725, val_acc=0.723


MSE KD: 100%|██████████| 391/391 [00:20<00:00, 18.86it/s]


Epoch 2: train_loss=0.804, acc=0.732, val_acc=0.729


MSE KD: 100%|██████████| 391/391 [00:20<00:00, 18.69it/s]


...

Epoch 8: train_loss=0.658, acc=0.780, val_acc=0.757


MSE KD: 100%|██████████| 391/391 [00:19<00:00, 20.28it/s]


Epoch 9: train_loss=0.644, acc=0.783, val_acc=0.763


MSE KD: 100%|██████████| 391/391 [00:20<00:00, 19.39it/s]


Epoch 10: train_loss=0.641, acc=0.786, val_acc=0.772
✅ Best val acc: 0.7781


**Комибинированная дистилляция**

In [ ]:
teacher = build_teacher().to(device)
student = SmallCNN().to(device)
teacher.load_state_dict(torch.load("/content/teacher.pth", map_location=device))
student.load_state_dict(torch.load("/content/student_base.pth", map_location=device))
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False
print("✅ Teacher & Student загружены")

✅ Teacher & Student загружены


In [ ]:
class MultiBlockTeacher(nn.Module):
    def __init__(self, teacher):
        super().__init__()
        # Разбиваем ResNet на блоки
        self.stage1 = nn.Sequential(teacher.conv1, teacher.bn1, teacher.relu, teacher.layer1)
        self.stage2 = teacher.layer2
        self.stage3 = teacher.layer3
        self.stage4 = teacher.layer4
        self.avgpool = teacher.avgpool
        self.fc = teacher.fc  # добавляем линейный классификатор

    def forward(self, x):
        f1 = self.stage1(x)
        f2 = self.stage2(f1)
        f3 = self.stage3(f2)
        f4 = self.stage4(f3)
        pooled = self.avgpool(f4)
        pooled = pooled.view(pooled.size(0), -1)
        logits = self.fc(pooled)  # теперь это выходы (batch, 10)
        return logits, [f1, f2, f3, f4]

teacher_multi = MultiBlockTeacher(teacher).to(device)
teacher_multi.eval()

# --- Расширим студента: добавим регрессоры для разных уровней ---
class StudentMultiReg(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, 10)

        # Регрессоры: каждый подгоняет фичи к размерности teacher'а
        self.reg1 = nn.Conv2d(32, 64, kernel_size=1)    # сопоставляется с layer1 (64)
        self.reg2 = nn.Conv2d(64, 128, kernel_size=1)   # сопоставляется с layer2 (128)
        self.reg3 = nn.Conv2d(128, 256, kernel_size=1)  # сопоставляется с layer3 (256)
        self.reg4 = nn.Conv2d(128, 512, kernel_size=1)  # сопоставляется с layer4 (512)

    def forward(self, x):
        f1 = self.relu(self.conv1(x))       # (B, 32, 32, 32)
        f2 = self.relu(self.conv2(self.pool(f1)))  # (B, 64, 16, 16)
        f3 = self.relu(self.conv3(self.pool(f2)))  # (B, 128, 8, 8)
        out = self.avgpool(f3)
        out = out.view(out.size(0), -1)
        logits = self.fc(out)
        # Прогон через регрессоры для MSE
        reg_feats = [
            self.reg1(f1),  # (B, 64, 32, 32)
            self.reg2(f2),  # (B, 128, 16, 16)
            self.reg3(f3),  # (B, 256, 8, 8)
            self.reg4(f3),  # (B, 512, 8, 8)
        ]
        return logits, reg_feats

student_multi = StudentMultiReg().to(device)

In [ ]:
# --- Комбинированный лосс ---
def combined_kd_loss(s_logits, y, t_logits, s_feats, t_feats, T=4.0, alpha=0.5, beta=0.5, gamma=0.1):
    # CrossEntropy
    ce = F.cross_entropy(s_logits, y)
    # KL между логитами (логитная дистилляция)
    p_t = F.softmax(t_logits / T, dim=1)
    p_s = F.log_softmax(s_logits / T, dim=1)
    kl = F.kl_div(p_s, p_t, reduction='batchmean') * (T * T)
    # MSE между фичами разных уровней
    mse_loss = 0.0
    for s_f, t_f in zip(s_feats, t_feats):
        # teacher feature map может быть другого spatial размера → уменьшим teacher
        if s_f.shape[-1] != t_f.shape[-1]:
            t_f = F.interpolate(t_f, size=s_f.shape[-2:], mode='bilinear', align_corners=False)
        mse_loss += F.mse_loss(s_f, t_f)
    mse_loss = mse_loss / len(s_feats)

    return alpha * ce + beta * kl + gamma * mse_loss

In [ ]:
def train_combined_kd(teacher_multi, student_multi, loader, optimizer, T=4.0, alpha=0.5, beta=0.5, gamma=0.1):
    student_multi.train()
    total_loss, total, correct = 0, 0, 0
    for x, y in tqdm(loader, desc="Combined KD"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        with torch.no_grad():
            t_logits, t_feats = teacher_multi(x)
        s_logits, s_feats = student_multi(x)

        loss = combined_kd_loss(s_logits, y, t_logits, s_feats, t_feats, T, alpha, beta, gamma)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)
        correct += (s_logits.argmax(1) == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out, _ = model(x)
        loss = F.cross_entropy(out, y)
        loss_sum += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, correct / total


# --- Тренировка ---
opt = torch.optim.SGD(student_multi.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5)

best_acc = 0
for epoch in range(10):
    train_loss, train_acc = train_combined_kd(teacher_multi, student_multi, train_loader, opt,
                                              T=4.0, alpha=0.4, beta=0.4, gamma=0.2)
    val_loss, val_acc = eval_model(student_multi, test_loader)
    scheduler.step()
    print(f"Epoch {epoch+1}: train_loss={train_loss:.3f}, acc={train_acc:.3f}, val_acc={val_acc:.3f}")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(student_multi.state_dict(), "student_combined_kd.pth")

print("✅ Best val acc:", best_acc)

Combined KD: 100%|██████████| 391/391 [00:27<00:00, 14.24it/s]


Epoch 1: train_loss=3.414, acc=0.307, val_acc=0.435


Combined KD: 100%|██████████| 391/391 [00:26<00:00, 14.59it/s]


Epoch 2: train_loss=2.330, acc=0.499, val_acc=0.568


Combined KD: 100%|██████████| 391/391 [00:26<00:00, 14.80it/s]


...

Epoch 8: train_loss=0.972, acc=0.745, val_acc=0.739


Combined KD: 100%|██████████| 391/391 [00:26<00:00, 14.89it/s]


Epoch 9: train_loss=0.921, acc=0.755, val_acc=0.750


Combined KD: 100%|██████████| 391/391 [00:26<00:00, 14.86it/s]


Epoch 10: train_loss=0.882, acc=0.762, val_acc=0.762
✅ Best val acc: 0.7616
